In [1]:
import pandas as pd 

In [2]:
# Inisialisasi Data
df = pd.read_csv('data_raw/messy_ecommerce_sales_data.csv', sep=',')
df.columns = df.columns.str.strip() # Menghapus whitespaces pada kolom/header tabel (jika ada)

In [3]:
# Menghapus Data
df.drop_duplicates(inplace=True)

# Mengambil semua kolom yang bertipe teks ('object') saja
text_coloumn = df.select_dtypes(include=['object']).columns

# Menghapus whitespace pada kolom teks secara keseluruhan
df[text_coloumn] = df[text_coloumn].apply(lambda x: x.str.strip())

df.head()

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,11/22/2024,Blender,Home,3,38,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,7/5/2025,Smartphone,Electronics,2,abd,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,12/23/2024,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,3/19/2025,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,10/20/2025,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51


In [4]:
# Order_Date
# Mengecek format yang tidak sesuai pada kolom 'Order_Date', lalu mengubah menjadi NaT atau kosong
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')

# Mengecek berapa banyak data yang berubah menjadi NaT atau kosong
print("Jumlah tanggal error:", df['Order_Date'].isna().sum())

# Menghapus baris pada kolom 'Order_Date' yang kosong
df.dropna(subset=['Order_Date'], inplace=True)

# Category
'''
Mengelompokan berdasarkan produk, lalu mengisi kolom kategori yang kosong
dengan kategori dari produk yang sama dari baris lain.
'''
df['Category'] = df['Category'].fillna(df.groupby('Product')['Category'].transform('first'))
df.tail(10)

Jumlah tanggal error: 2


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
91,191,Customer_191,ORD-41810,2025-07-18,Fiction,Books,2,881.02,Cash on Delivery,Returned,1762.040
93,193,Customer_193,ORD-42475,2025-06-06,Basketball,sports,NaN,522.02,PayPal,Shipped,NaN
94,194,Customer_194,ORD-69764,2025-02-10,Lamp,Home,5,944.54,Cash on Delivery,Returned,4722.700
95,195,Customer_195,ORD-82876,2025-06-19,Biography,Books,5,817.46,Cash on Delivery,Returned,4087.300
96,196,Customer_196,ORD-78384,2024-12-23,Vacuum,Home,NaN,abd,PayPal,Delivered,NaN
97,197,Customer_197,ORD-79139,2025-06-23,Blender,Home,1,160.16,PayPal,Cancelled,160.160
98,198,Customer_198,ORD-14608,2025-07-27,Vacuum,electronic,2,497.01,Cash on Delivery,Shipped,994.020
99,199,Customer_199,ORD-82922,2025-01-22,Blender,Home,5,372.28,Credit Card,Shipped,1861.400
100,175,Customer_175,ORD-56651,2025-02-24,Headphones,Electronics,1,111.36,Credit Card,Processing,77.952
101,142,Customer_142,ORD-69018,2025-10-30,Shoes,Clothing,5,645.26,Credit Card,Shipped,3226.300


In [5]:
# Quantity
# Mengubah ke tipe data str -> ubah ke kapital -> mengganti huruf dengan string kosong (jika ada huruf yang typo)
df['Quantity'] = df['Quantity'].astype(str).str.upper().str.replace(r'[A-Z]', '', regex=True)

# Mengubah ke dalam bentuk numerik, yang error akan jadi NaN
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

# Isi nilai NaN (missing values) dengan nilai median, setelahnya ubah ke int
median_quantity = df['Quantity'].median()
df['Quantity'] = df['Quantity'].fillna(median_quantity).astype(int)

# Mengubah Format Data yang salah (negatif -> positif)
df['Quantity'] = df['Quantity'].abs()

# Price
# Memeriksa data yang tidak sesuai dan mengubahnya menjadi NaN
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
# Mengisi harga yang kosong dengan menyalin median harga dari produk yang sama
df['Price'] = df['Price'].fillna(df.groupby('Product')['Price'].transform('median'))

df.tail(12)

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
89,189,Customer_189,ORD-30627,2025-08-14,Lamp,Home,1,814.68,Cash on Delivery,Returned,814.680
90,190,Customer_190,ORD-38469,2025-06-29,Vacuum,Home,1,920.37,Bank Transfer,Processing,920.370
91,191,Customer_191,ORD-41810,2025-07-18,Fiction,Books,2,881.02,Cash on Delivery,Returned,1762.040
93,193,Customer_193,ORD-42475,2025-06-06,Basketball,sports,3,522.02,PayPal,Shipped,NaN
94,194,Customer_194,ORD-69764,2025-02-10,Lamp,Home,5,944.54,Cash on Delivery,Returned,4722.700
95,195,Customer_195,ORD-82876,2025-06-19,Biography,Books,5,817.46,Cash on Delivery,Returned,4087.300
96,196,Customer_196,ORD-78384,2024-12-23,Vacuum,Home,3,556.46,PayPal,Delivered,NaN
97,197,Customer_197,ORD-79139,2025-06-23,Blender,Home,1,160.16,PayPal,Cancelled,160.160
98,198,Customer_198,ORD-14608,2025-07-27,Vacuum,electronic,2,497.01,Cash on Delivery,Shipped,994.020
99,199,Customer_199,ORD-82922,2025-01-22,Blender,Home,5,372.28,Credit Card,Shipped,1861.400


In [6]:
# Payment Method
# Mengecek jumlah data yang hilang (jika ada)
print('Jumlah data hilang (payment method):', df['Payment_Method'].isna().sum())

# Status
# Mengecek jumlah data yang hilang (jika ada)
print('Jumlah data hilang (status):', df['Status'].isna().sum())



Jumlah data hilang (payment method): 0
Jumlah data hilang (status): 0
